# Live LLM Arena with Eden AI

Send the same prompt to four LLMs in parallel, watch them stream side-by-side, then pick the winner (human vote or LLM-as-judge).

Same API, same key, same response format — only the `model` string changes per call. That is the point of this notebook.

**Prerequisites:** an Eden AI API key (set as `EDENAI_API_KEY` env var).

In [16]:
%pip install --quiet aiohttp ipywidgets nest_asyncio python-dotenv

Note: you may need to restart the kernel to use updated packages.


## 1. Configuration

Each entry is one model in the arena. Add or remove rows freely — the grid auto-resizes.

In [17]:
import base64
import json
import os

from dotenv import load_dotenv
from IPython.display import HTML, display

load_dotenv(override=True)

EDENAI_API_KEY = os.environ.get("EDENAI_API_KEY")
if not EDENAI_API_KEY:
    raise RuntimeError("Set EDENAI_API_KEY (env var or .env file). Get one at https://app.edenai.run")
EDENAI_URL = "https://api.edenai.run/v3/llm/chat/completions"

MODELS = [
    {"label": "Claude",   "model": "anthropic/claude-sonnet-4-5"},
    {"label": "GPT",      "model": "openai/gpt-4"},
    {"label": "DeepSeek", "model": "deepseek/deepseek-chat"},
    {"label": "GLM",      "model": "deepinfra/zai-org/GLM-5.1"},
]

JUDGE_MODEL = "anthropic/claude-sonnet-4-5"


def _is_sandbox(jwt: str) -> bool:
    try:
        payload_b64 = jwt.split(".")[1]
        payload_b64 += "=" * (-len(payload_b64) % 4)
        return json.loads(base64.urlsafe_b64decode(payload_b64)).get("type") == "sandbox_api_token"
    except Exception:
        return False


if _is_sandbox(EDENAI_API_KEY):
    display(HTML(
        '<div style="background:#fff3cd;border-left:4px solid #ffc107;padding:10px 14px;'
        'border-radius:4px;font-family:sans-serif;font-size:13px;margin:6px 0;">'
        '<b>⚠ Sandbox key detected.</b> All models will return the same mocked response, '
        'so the arena comparison will look identical across panels. '
        'Use a production key from <a href="https://app.edenai.run" target="_blank">app.edenai.run</a> '
        'to see real differences.</div>'
    ))

## 2. Streaming caller

Eden AI's `/v3/llm/chat/completions` is OpenAI-compatible, including SSE streaming. Each chunk arrives as `data: { ... }` with the new token in `choices[0].delta.content`. We pipe that into an `ipywidgets.Output` so each model's pane fills in live.

In [18]:
import json
import time

import aiohttp


def _header_html(label, model, status="idle", status_color="#6c757d", latency=None):
    lat = f" · {latency:.2f}s" if latency is not None else ""
    return (
        '<div style="display:flex;justify-content:space-between;align-items:center;'
        'font-family:sans-serif;padding:4px 2px;">'
        f'  <div>'
        f'    <span style="font-weight:600;font-size:14px;">{label}</span>'
        f'    <span style="color:#888;font-size:11px;margin-left:6px;">{model}</span>'
        '  </div>'
        '  <div>'
        f'    <span style="background:{status_color};color:white;padding:3px 10px;'
        f'border-radius:10px;font-size:11px;font-weight:600;">{status}{lat}</span>'
        '  </div>'
        '</div>'
    )


async def stream_model(session, model_cfg, prompt, output_widget, header_widget):
    headers = {
        "Authorization": f"Bearer {EDENAI_API_KEY}",
        "Content-Type": "application/json",
    }
    payload = {
        "model": model_cfg["model"],
        "messages": [{"role": "user", "content": prompt}],
        "stream": True,
    }

    output_widget.clear_output()
    full_text = []
    first_token_at = None
    start = time.perf_counter()
    header_widget.value = _header_html(model_cfg["label"], model_cfg["model"], "streaming…", "#17a2b8")

    async with session.post(EDENAI_URL, headers=headers, json=payload) as resp:
        if resp.status != 200:
            header_widget.value = _header_html(model_cfg["label"], model_cfg["model"], "error ✗", "#dc3545")
            with output_widget:
                print(f"[error {resp.status}] {await resp.text()}")
            return {"label": model_cfg["label"], "text": "", "first_token": None, "total": None}

        async for raw in resp.content:
            line = raw.decode("utf-8").strip()
            if not line.startswith("data:"):
                continue
            data = line[5:].strip()
            if data == "[DONE]":
                break
            try:
                chunk = json.loads(data)
            except json.JSONDecodeError:
                continue
            token = chunk.get("choices", [{}])[0].get("delta", {}).get("content", "")
            if not token:
                continue
            if first_token_at is None:
                first_token_at = time.perf_counter() - start
            full_text.append(token)
            with output_widget:
                print(token, end="")
            header_widget.value = _header_html(
                model_cfg["label"], model_cfg["model"],
                "streaming…", "#17a2b8", time.perf_counter() - start,
            )

    total = time.perf_counter() - start
    header_widget.value = _header_html(model_cfg["label"], model_cfg["model"], "done ✓", "#28a745", total)
    return {
        "label": model_cfg["label"],
        "text": "".join(full_text),
        "first_token": first_token_at,
        "total": total,
    }

## 3. Arena UI

A 2-column grid of streaming panes plus a prompt box, fight button, and per-model vote buttons.

In [19]:
from ipywidgets import (
    Button, GridBox, HBox, HTML as HTMLWidget, Layout, Output, Textarea, VBox,
)
from IPython.display import display

prompt_box = Textarea(
    placeholder="Type a prompt for all models, then press Fight…",
    layout=Layout(width="100%", height="60px"),
)
fight_btn = Button(description="⚔  Fight", button_style="primary")
clear_btn = Button(description="Clear panels")

panel_layout = Layout(
    border="1px solid #ddd",
    padding="8px",
    height="220px",
    overflow="auto",
)
panels = [Output(layout=panel_layout) for _ in MODELS]
headers = [HTMLWidget(value=_header_html(m["label"], m["model"])) for m in MODELS]
panel_blocks = [
    VBox([headers[i], panels[i]], layout=Layout(border="1px solid #eee", padding="4px", border_radius="4px"))
    for i in range(len(MODELS))
]
grid = GridBox(
    panel_blocks,
    layout=Layout(grid_template_columns="repeat(2, 1fr)", grid_gap="8px"),
)

vote_btns = [Button(description=f"Vote {m['label']}") for m in MODELS]
judge_btn = Button(description="🧑‍⚖️ LLM judge")
vote_bar = HBox([*vote_btns, judge_btn])

scoreboard = Output()
judge_out = Output()

display(VBox([
    prompt_box,
    HBox([fight_btn, clear_btn]),
    grid,
    vote_bar,
    scoreboard,
    judge_out,
]))

## 4. Wire it up

Fight button → fan out to all models in parallel via `asyncio.gather`. Vote button increments a score; LLM-judge button sends every response to a referee model.

In [20]:
import asyncio

import nest_asyncio
from IPython.display import HTML, clear_output, display

nest_asyncio.apply()

scores = {m["label"]: 0 for m in MODELS}
last_round = {}


def render_scoreboard():
    max_score = max(scores.values()) if scores.values() else 0
    rows = ""
    for label, n in sorted(scores.items(), key=lambda kv: -kv[1]):
        width = int(n / max_score * 200) if max_score else 0
        rows += (
            '<div style="margin:4px 0;display:flex;align-items:center;gap:8px;'
            'font-family:sans-serif;font-size:13px;">'
            f'  <div style="width:80px;">{label}</div>'
            f'  <div style="height:18px;width:{width}px;background:#007bff;'
            'border-radius:4px;transition:width 0.3s;"></div>'
            f'  <div style="margin-left:6px;font-weight:600;">{n}</div>'
            '</div>'
        )
    with scoreboard:
        clear_output()
        display(HTML(
            f'<div><b style="font-family:sans-serif;font-size:13px;">Scoreboard</b>{rows}</div>'
        ))


def _highlight_winner(label):
    for i, m in enumerate(MODELS):
        if m["label"].lower() == label.lower():
            panel_blocks[i].layout.border = "2px solid #28a745"
        else:
            panel_blocks[i].layout.border = "1px solid #eee"


def _reset_borders():
    for pb in panel_blocks:
        pb.layout.border = "1px solid #eee"


async def run_round(prompt):
    _reset_borders()
    async with aiohttp.ClientSession() as session:
        tasks = [
            stream_model(session, MODELS[i], prompt, panels[i], headers[i])
            for i in range(len(MODELS))
        ]
        return await asyncio.gather(*tasks)


def on_fight(_):
    prompt = prompt_box.value.strip()
    if not prompt:
        return
    judge_out.clear_output()
    results = asyncio.run(run_round(prompt))
    last_round.clear()
    last_round.update({r["label"]: r["text"] for r in results})


def on_clear(_):
    for i, m in enumerate(MODELS):
        panels[i].clear_output()
        headers[i].value = _header_html(m["label"], m["model"])
    _reset_borders()
    judge_out.clear_output()
    last_round.clear()


def make_vote_handler(label):
    def handler(_):
        scores[label] += 1
        _highlight_winner(label)
        render_scoreboard()
    return handler


async def call_judge(prompt, responses):
    blocks = "\n\n".join(f"### {label}\n{text}" for label, text in responses.items())
    judge_prompt = (
        f"Original question:\n{prompt}\n\n"
        f"Candidate answers:\n{blocks}\n\n"
        "Pick the single best answer for accuracy and clarity. "
        "First write one sentence of reasoning. "
        "Then on a new line, output ONLY the label of the winner (one word)."
    )
    async with aiohttp.ClientSession() as session:
        headers_dict = {
            "Authorization": f"Bearer {EDENAI_API_KEY}",
            "Content-Type": "application/json",
        }
        payload = {
            "model": JUDGE_MODEL,
            "messages": [{"role": "user", "content": judge_prompt}],
        }
        async with session.post(EDENAI_URL, headers=headers_dict, json=payload) as r:
            data = await r.json()
            return data["choices"][0]["message"]["content"].strip()


def on_judge(_):
    if not last_round:
        return
    response = asyncio.run(call_judge(prompt_box.value.strip(), last_round))
    lines = [ln.strip() for ln in response.splitlines() if ln.strip()]
    winner_line = lines[-1] if lines else response
    reasoning = " ".join(lines[:-1]) if len(lines) > 1 else ""

    chosen = None
    for label in scores:
        if label.lower() in winner_line.lower():
            scores[label] += 1
            chosen = label
            break

    with judge_out:
        clear_output()
        reasoning_html = (
            f'<div style="font-family:sans-serif;font-size:12px;color:#555;margin-top:4px;">'
            f'{reasoning}</div>'
        ) if reasoning else ""
        winner_html = (
            f'<div style="font-family:sans-serif;font-size:13px;">'
            f'<b>🧑‍⚖️ Judge picked:</b> {chosen or winner_line}</div>'
        )
        display(HTML(
            f'<div style="background:#f8f9fa;padding:10px 12px;border-radius:4px;'
            f'border-left:4px solid #28a745;">{winner_html}{reasoning_html}</div>'
        ))

    if chosen:
        _highlight_winner(chosen)
        render_scoreboard()


fight_btn.on_click(on_fight)
clear_btn.on_click(on_clear)
judge_btn.on_click(on_judge)
for btn, m in zip(vote_btns, MODELS):
    btn.on_click(make_vote_handler(m["label"]))

## 5. Try these prompts

- **Reasoning:** *A bat and a ball cost \$1.10 in total. The bat costs \$1 more than the ball. How much does the ball cost?*
- **Creative:** *Write a 4-line poem about a lighthouse, but every line must start with the letter S.*
- **Code:** *Write a Python one-liner that returns the longest word in a string, ties broken by leftmost.*
- **Multilingual:** *Translate "knowledge is power" into Japanese, then explain the literal meaning of each character.*

## 6. Customize

- **More models:** add rows to `MODELS` — the grid auto-expands. Try `google/gemini-2.5-flash`, `mistral/mistral-large-latest`, or `cohere/command-a-03-2025`. The full catalog is at `GET /v3/llm/models`.
- **Persist scores:** dump `scores` to JSON between sessions to keep a running leaderboard.